In [1]:
import numpy as np
from ansys.mapdl.core import launch_mapdl
exec_loc = "/Program Files (x86)/ANSYS Inc/v222/ansys/bin/winx64/ansys222.exe" #change to your executable location
mapdl = launch_mapdl(exec_loc,additional_switches='-smp',override=True)#

In [2]:
#!!!!!!!!!!!!!!!!
#!  PARAMETERS
#!!!!!!!!!!!!!!!!
R=12.6051E-3
H=60E-3
NV=40
NC=37
D_SHELIX=0.97804E-3
AA=0.19453E-3
BB=0.19376E-3
LOOP_NO = 4
PITCH=H/LOOP_NO
PI = np.pi
DELTA_TH = 2*PI/NV
DELTA_H = H/(NC-1)
TH_SHELIX = D_SHELIX/R
NOBUCK = 10

In [3]:
mapdl.prep7()

*** MAPDL - ENGINEERING ANALYSIS SYSTEM  RELEASE 2022 R2          22.2     ***
 Ansys Mechanical Enterprise Academic Research     
 00000000  VERSION=WINDOWS x64   17:25:47  JUN 27, 2023 CP=      0.094

                                                                               



          ***** MAPDL ANALYSIS DEFINITION (PREP7) *****

In [4]:
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!  CROSS SECTION OF THE BIG HELIX
#!!  AT THE END SECTION IS SAVED AND
#!!  UNNECESSARY DATA IS DELETED
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
mapdl.et(1,'PLANE82')
mapdl.clocal(11,1,"","","","","","",2)
mapdl.csys(11)
RR = 1.2269E-3
COUNT = 1
for TH in range(0,181,15):
    mapdl.k(COUNT,RR,TH,0)
    COUNT = COUNT + 1
    
mapdl.flst(3,13,3)
SPLINE_COUNT = 1
for I in range(1,14):
    mapdl.fitem(3,SPLINE_COUNT)
    SPLINE_COUNT = SPLINE_COUNT + 1

mapdl.bsplin("",'P51X')
mapdl.csys(0)
mapdl.l(1,13)
mapdl.lesize("All","","",40,"",1,"","",1)
mapdl.al(1,2)
mapdl.amesh("All")
mapdl.esel("All")

mapdl.eplot("All") #plot element cross section of big helix

mapdl.secwrite("big_helix")
mapdl.aclear("All")
mapdl.adele("All","","",1)

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

DELETE   ALL   SELECTED AREAS.

     DELETED    1 AREAS,     2 LINES,      2 KEYPOINTS

In [5]:
#!!!!!!!!!!!!!!!!!!!!!!
#!!  MAIN CODE STARTS
#!!!!!!!!!!!!!!!!!!!!!!

mapdl.et(1,'BEAM188') #ELEMENT TYPE 3D 2-node beam element
mapdl.mptemp("","","","","","","")
mapdl.mptemp(1,0)
mapdl.mpdata("EX",1,"",91900000000)
mapdl.mpdata("PRXY",1,"",0.17) #Define material properties
mapdl.sectype(1,"BEAM","RECT","",0) #Define rectangular cross section for beam elements
mapdl.secoffset("CENT")
mapdl.secdata(AA,BB,0,0,0,0,0,0,0,0,0,0) #Defines size of cross section

mapdl.sectype(2,"BEAM","MESH") #ELLIPSOIDAL Cross SECTION
mapdl.secoffset("ORIG","","","")
mapdl.secread("big_helix","SECT","MESH")


USER BEAM SECTION DATA WAS READ FROM FILE= big_helix.SECT                                                                                                                                                                                                                                                      
                     
   SECTION ID NUMBER IS:            2
   BEAM SECTION TYPE IS:     User Mesh       
   BEAM SECTION NAME IS:             
   COMPUTED BEAM SECTION DATA SUMMARY:
    Area                 = 0.47290E-05
    Iyy                  = 0.19895E-11
    Iyz                  = 0.21335E-19
    Izz                  = 0.17797E-11
    Warping Constant     = 0.25245E-19
    Torsion Constant     = 0.31825E-11
    Centroid Y           = 0.43506E-12
    Centroid Z           = 0.10414E-02
    Shear Center Y       = 0.14088E-11
    Shear Center Z       = 0.10712E-02
    Shear Correction-xy  = 0.80771    
    Shear Correction-yz  =-0.62956E-07
    Shear Correction-xz  = 0.83144    
      

In [6]:
#!!!!!!!!!!!!!!!!!!!!!!
#!!  VERTICAL BEAMS
#!!!!!!!!!!!!!!!!!!!!!!
COUNT=1
TH=0
for II in range(1,NV+1):
    mapdl.k(COUNT,R*np.cos(TH),R*np.sin(TH),0)
    TH = TH + DELTA_TH
    COUNT = COUNT + 1

TH=0
for II in range(1,NV+1):
    mapdl.k(COUNT,R*np.cos(TH),R*np.sin(TH),H)
    TH = TH + DELTA_TH
    COUNT = COUNT + 1

In [7]:
#!!!!!!!!!!!!!!!!!!!!!!!!!!!
#!!  CIRCUMFERENTIAL BEAMS
#!!!!!!!!!!!!!!!!!!!!!!!!!!!
COUNT_LINE=1
for II in range(1,NV+1):
    mapdl.l(COUNT_LINE,COUNT_LINE+NV)
    COUNT_LINE = COUNT_LINE + 1

HH=0
for II in range(1,NC+1):
    mapdl.k(COUNT,0,0,HH)
    mapdl.circle(COUNT,R)
    COUNT = COUNT + 5
    COUNT_LINE = COUNT_LINE + 4
    HH = HH + DELTA_H
    
mapdl.lplot("All")


ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

In [8]:
#!!!!!!!!!!!!!!!!!!!!!
#!!  SMALL HELICES 1st direction
#!!!!!!!!!!!!!!!!!!!!!
TH0 = TH_SHELIX/2
TH1 = -TH_SHELIX/2
HH=0

for JJ in range(1,int(NV/2)+1):
    SPLINE_INIT = COUNT
    HH=0
    for II in range(1,NC+1):
        mapdl.k(COUNT,R*np.cos(TH0),R*np.sin(TH0),HH)
        COUNT = COUNT + 1
        TH0 = TH0 + DELTA_TH
        HH = HH + DELTA_H 
    mapdl.flst(3,NC,3)
    for II in range(1,NC+1):
        mapdl.fitem(3,SPLINE_INIT)
        SPLINE_INIT = SPLINE_INIT + 1
    mapdl.bsplin("",'P51X')
    
    COUNT_LINE = COUNT_LINE + 1
    SPLINE_INIT = COUNT
    HH=0
    for II in range(1,NC+1):
        mapdl.k(COUNT,R*np.cos(TH1),R*np.sin(TH1),HH)
        COUNT = COUNT + 1
        TH1 = TH1 + DELTA_TH
        HH = HH + DELTA_H
    mapdl.flst(3,NC,3)
    
    for II in range(1,NC+1):
        mapdl.fitem(3,SPLINE_INIT)
        SPLINE_INIT = SPLINE_INIT + 1
    mapdl.bsplin("",'P51X')
    
    COUNT_LINE = COUNT_LINE + 1
    TH0 = TH_SHELIX/2+(2*DELTA_TH)*JJ
    TH1 = -TH_SHELIX/2+(2*DELTA_TH)*JJ
    
mapdl.lplot("All")

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

In [9]:
#!!!!!!!!!!!!!!!!!!!!!
#!!  SMALL HELICES 2nd direction
#!!!!!!!!!!!!!!!!!!!!!
TH0 = TH_SHELIX/2
TH1 = -TH_SHELIX/2
HH=0

for JJ in range(1,int(NV/2)+1):
    SPLINE_INIT = COUNT
    HH=0
    for II in range(1,NC+1):
        mapdl.k(COUNT,R*np.cos(TH0),R*np.sin(TH0),HH)
        COUNT = COUNT + 1
        TH0 = TH0 - DELTA_TH #sign switches
        HH = HH + DELTA_H 
    mapdl.flst(3,NC,3)
    for II in range(1,NC+1):
        mapdl.fitem(3,SPLINE_INIT)
        SPLINE_INIT = SPLINE_INIT + 1
    mapdl.bsplin("",'P51X')
    
    COUNT_LINE = COUNT_LINE + 1
    SPLINE_INIT = COUNT
    HH=0
    for II in range(1,NC+1):
        mapdl.k(COUNT,R*np.cos(TH1),R*np.sin(TH1),HH)
        COUNT = COUNT + 1
        TH1 = TH1 - DELTA_TH #sign switches
        HH = HH + DELTA_H
    mapdl.flst(3,NC,3)
    
    for II in range(1,NC+1):
        mapdl.fitem(3,SPLINE_INIT)
        SPLINE_INIT = SPLINE_INIT + 1
    mapdl.bsplin("",'P51X')
    
    COUNT_LINE = COUNT_LINE + 1
    TH0 = TH_SHELIX/2+(2*DELTA_TH)*JJ
    TH1 = -TH_SHELIX/2+(2*DELTA_TH)*JJ
    
mapdl.lplot("All")

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

In [10]:
#!!!!!!!!!!!!!!!!!!!!!
#!!  Large HELICE
#!!!!!!!!!!!!!!!!!!!!!

NUM_KP = mapdl.get('NUM_KP',"KP",0,"NUM","MAXD")
COUNT = NUM_KP + 1
HH = 0
TH0 = 0
DELTA_H_BIGHELIX = PITCH/NV
SPLINE_INIT = COUNT

for II in range(1,(NV)*LOOP_NO+2):
    mapdl.k(COUNT,R*np.cos(TH0),R*np.sin(TH0),HH)
    TH0 = TH0 - DELTA_TH
    HH = HH + DELTA_H_BIGHELIX
    COUNT = COUNT + 1
mapdl.flst(3,(NV)*LOOP_NO+1,3)

for II in range(1,(NV)*LOOP_NO+2):
    mapdl.fitem(3,SPLINE_INIT)
    SPLINE_INIT = SPLINE_INIT + 1
mapdl.bsplin("",'P51X')
mapdl.lplot("All")

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

In [11]:
mapdl.lsel("All")
mapdl.btol("1e-10")
mapdl.lovlap("All")
print("Beams merged")

Beams merged


In [12]:
mapdl.lsel("All")
mapdl.lmesh("All")
mapdl.eplot(show_edges=True, smooth_shading=True, show_node_numbering=False)

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

In [13]:
mapdl.exit()